# Colab / Kaggle training

Runs the same training code and `configs/config.yaml` as the laptop. Use it for extra seeds or ablations when the local GPU is busy. Research demo, not for clinical use.

1. Runtime > Change runtime type > GPU.
2. Set `REPO` to your GitHub URL.
3. Run all. Checkpoints and run JSON are copied to Google Drive.

In [ ]:
REPO = 'https://github.com/<your-account>/leukemia-classifier.git'  # set this
!git clone $REPO repo
%cd repo
!pip -q install timm scikit-learn pandas pyyaml datasets huggingface_hub torchstain grad-cam captum

In [ ]:
# Paths for Colab: data and checkpoints on the VM, results copied to Drive at the end
import yaml, pathlib
cfg = yaml.safe_load(open('configs/config.yaml'))
cfg['paths'].update(data_root='/content/cnmc', checkpoints='/content/checkpoints')
cfg['data']['num_workers'] = 2
cfg['train']['amp'] = True  # T4/A100 support fp16 correctly; the fp32 rule is only for GTX 16-series
yaml.safe_dump(cfg, open('configs/config.yaml', 'w'), sort_keys=False)

In [ ]:
!python -m src.download && python -m src.split && python -m pytest -q tests/test_split_leakage.py

In [ ]:
# Example: one extra ensemble seed and one ablation
!python -m src.train --name effb0_main_s47 --set seed=47
!python -m src.train --name abl_scalejit_s43 --set seed=43 train.augment=geometric_scale

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/leukemia-runs
!cp -r reports/runs reports/preds /content/checkpoints /content/drive/MyDrive/leukemia-runs/